In [15]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    mean_absolute_error,root_mean_squared_error,mean_squared_error,r2_score,make_scorer
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score,KFold
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)


In [16]:

df = pd.read_csv('Copia de data_merged.csv')



In [17]:

def aux(temp):
    if temp<=80:
        return 'lento'
    elif temp <=120:
        return 'moderado'
    return 'rapido'

df['duration_ms']= df['duration_ms']/60000
df['duration_ms_ar'] = df['duration_ms_ar']/60000
df['duration_ms_yr'] = df['duration_ms_yr']/60000

df['energy_ratio_yr'] = df['energy_yr']*df['acousticness_yr']
df['electro_focus_yr'] = (1-df['acousticness_yr'])*df['energy_yr']*df['instrumentalness_yr']
df['vocal_energy_yr'] = (1-df['instrumentalness_yr'])*df['energy_yr']

df['energy_ratio_'] = df['energy']*df['acousticness']
df['electro_focus'] = (1-df['acousticness'])*df['energy']*df['instrumentalness']
df['vocal_energy'] = (1-df['instrumentalness'])*df['energy']

df['energy_ratio_ar'] = df['energy_ar']*df['acousticness_ar']
df['electro_focus_ar'] = (1-df['acousticness_ar'])*df['energy_ar']*df['instrumentalness_ar']
df['vocal_energy_ar'] = (1-df['instrumentalness_ar'])*df['energy_ar']


df['temp_range'] = df['tempo_yr'].apply(aux)
df = pd.get_dummies(df,columns=['temp_range'],prefix='temp')
df[['temp_moderado','temp_rapido']] = df[['temp_moderado','temp_rapido']].astype(int)

In [18]:
X = df.drop('popularity',axis=1)
y=df['popularity']

In [19]:

selector = SelectKBest(score_func=f_regression,k=20)
X_new = selector.fit_transform(X,y)
select = selector.get_feature_names_out()
select

X = X[select]

In [20]:
X_train,X_test,y_train,y_test=train_test_split(X,y)


In [21]:
def log_trannsform(y):
    return np.log1p(y)
def inverse_log_trannsform(y):
    return np.expm1(y)
pipeline = Pipeline(
    [
        (
            'scaler',RobustScaler()
        ),
        (
            'mlp',MLPRegressor(
                hidden_layer_sizes=(128,64,32),
                activation='relu',
                solver='adam',
                alpha=0.001,
                batch_size=32,
                learning_rate_init=0.001,
                max_iter=1000,
                early_stopping=True,
                validation_fraction=0.1,
                verbose=42
                            
                )
        )
    ]
)
cv = KFold(n_splits=3,shuffle=True)
y_log = log_trannsform(y)
scores = cross_val_score(
pipeline,
X_train,
y_train,
cv=cv,
scoring='neg_mean_absolute_error',
n_jobs=-1
)



Iteration 1, loss = 51.81940478
Iteration 1, loss = 51.80733942
Validation score: 0.838890
Iteration 1, loss = 52.98085077
Validation score: 0.838368
Validation score: 0.835514
Iteration 2, loss = 38.29396948
Validation score: 0.843187
Iteration 2, loss = 37.49287206
Validation score: 0.839689
Iteration 2, loss = 37.89452952
Validation score: 0.839638
Iteration 3, loss = 37.40734064
Validation score: 0.845504
Iteration 3, loss = 36.80313020
Validation score: 0.836824
Iteration 3, loss = 37.09542999
Validation score: 0.841513
Iteration 4, loss = 36.88638791
Validation score: 0.847505
Iteration 4, loss = 36.29568424
Validation score: 0.839318
Iteration 4, loss = 36.53951248
Validation score: 0.846081
Iteration 5, loss = 36.49620280
Validation score: 0.843908
Iteration 5, loss = 35.85016332
Validation score: 0.846871
Iteration 5, loss = 35.99315204
Validation score: 0.844589
Iteration 6, loss = 36.11136256
Validation score: 0.849742
Iteration 6, loss = 35.43011319
Iteration 6, loss = 35.7